# Tidal Hydrodynamic Screening Model — Complete Explainer

This notebook provides an in-depth, function-by-function walkthrough of the **tidal-current energy assessment model**. We'll cover:

1. **Data pipeline** — the external datasets (GEBCO, GOT4.10c, shapefiles) that feed the model
2. The **Arakawa C-grid** and how the ocean is represented
3. **Bathymetry** loading, regridding, and land masking
4. **Coriolis** and the staggered-grid interpolation utilities
5. **Tidal forcing** — synthetic, GOT4.10c, FES2014, and TPXO9
6. The **shallow-water solver** — governing equations and time-stepping
7. The **CFL stability condition** and adaptive time steps
8. **Running** the model, callbacks, and mass-conservation checks
9. **Output** — NetCDF, GeoTIFF, hotspot GeoJSON
10. **Parameter sensitivity** — how each knob changes the result
11. **Validation tests** — seiche period, tidal channel, conservation

Each section includes runnable Python examples so you can experiment interactively.

In [ ]:
import sys
import os
import numpy as np
from pathlib import Path

# Ensure the src/model package is on the path — resolve from this notebook's location
try:
    # __file__ is available in regular Python; in Jupyter use the working directory
    _notebook_dir = Path(os.getcwd()).resolve()
except NameError:
    _notebook_dir = Path.cwd()

# Walk up to find the project root (contains src/, docs/, README.md)
PROJECT_ROOT = _notebook_dir
for _ in range(5):
    if (PROJECT_ROOT / 'src' / 'model').is_dir() and (PROJECT_ROOT / 'README.md').is_file():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print(f'Project root: {PROJECT_ROOT}')

# Import model components
from model.grid import StructuredGrid
from model.solver import ShallowWaterSolver, G, RHO_SEAWATER
from model.forcing import (
    ASTRO_FREQUENCIES,
    TidalConstituent,
    TidalBoundary,
    make_synthetic_tidal_boundary,
    build_tidal_boundary,
)
from model.utils import (
    coriolis,
    cfl_timestep,
    interpolate_to_u,
    interpolate_to_v,
    v_at_u_pts,
    u_at_v_pts,
    velocity_at_centres,
    speed,
    power_density,
)
from model.output import (
    create_results_dataset,
    write_netcdf,
    write_mean_power_geotiff,
    write_hotspots_geojson,
)

print("All imports successful. Model ready.")

---
## 1. Data Pipeline & External Datasets

Before diving into the model internals, it's essential to understand what data drives it. The model consumes **three categories** of external data, each from a different source:

### 1.1 The Three Pillars

| Dataset | Type | Source | Size | Required? |
|---------|------|--------|------|-----------|
| **GEBCO 2024** | Bathymetry (seabed depth) | GEBCO Compilation Group | ~2.7 GB (global NetCDF) | No — model falls back to a synthetic flat grid |
| **GOT4.10c / FES2014 / TPXO9** | Tidal harmonic constants | NASA / AVISO / OSU | 44 MB – 4 GB | No — model falls back to synthetic M2 forcing |
| **GADM + OSM shapefiles** | Land/coastline polygons | GADM / Geofabrik | ~40 MB (zipped) | No — elevation > 0 fallback used if absent |

Each dataset enters the pipeline at a specific stage:

```
GEBCO NetCDF ──→ load_gebco() ──→ regrid_bathymetry() ──→ elevation_to_depth() ──┐
                                                                                 │
GADM .shp ────→ build_land_mask() ────────────────────────────────────────────────┤
                                                                                 ├──→ StructuredGrid
GOT/FES/TPXO ──→ read_*_constituents() ──→ build_tidal_boundary() ──→ solver ◄───┘
NetCDF
```

### 1.2 GEBCO Bathymetry — The Seabed

**GEBCO** (General Bathymetric Chart of the Oceans) is the authoritative global terrain model of the ocean floor. The 2024 release provides:

- **Resolution:** 15 arc-seconds (~450 m at the equator)
- **Coverage:** Global, land + ocean
- **Format:** Single NetCDF file (`GEBCO_2024.nc`) with `lon`, `lat`, and `elevation` variables
- **Convention:** Elevation is **positive up** (land > 0, seabed < 0). The model inverts this to depth-positive-down.
- **Licence:** Free, but requires accepting terms at [gebco.net](https://www.gebco.net/data_and_products/gridded_bathymetry_data/gebco_2024/)

For the Philippine domain (116°–130°E, 4°–22°N), the GEBCO subset is ~250 MB — only ~10% of the full global file. You can pre-clip it with `ncks` or let the model's `load_gebco()` clip on-the-fly.

**Why bathymetry matters for tidal power:** The seabed shape controls both wave propagation speed ($c = \sqrt{gh}$) and bottom friction ($\tau_b \propto |U|U / h$). Deep channels transmit tidal energy efficiently; shallow shelves dissipate it. Narrow passages between islands funnel flow — the fundamental reason straits are hotspots.

In [ ]:
# Explore the GEBCO data loading pipeline (without needing the actual file)
# If you have GEBCO downloaded, point to it here:
GEBCO_PATH = PROJECT_ROOT / 'data' / 'gebco_bathymetry'

# Check what's available
gebco_files = list(GEBCO_PATH.glob('*.nc')) if GEBCO_PATH.exists() else []
if gebco_files:
    import xarray as xr
    ds = xr.open_dataset(gebco_files[0], decode_times=False)
    print(f"GEBCO file: {gebco_files[0].name}")
    print(f"Variables: {list(ds.data_vars)}")
    print(f"Dimensions: {dict(ds.sizes)}")
    print(f"Lon range: {float(ds.lon.min()):.1f} – {float(ds.lon.max()):.1f}°")
    print(f"Lat range: {float(ds.lat.min()):.1f} – {float(ds.lat.max()):.1f}°")
    elev = ds.elevation.values
    print(f"Elevation range: {elev.min():.0f} – {elev.max():.0f} m (positive up)")
    print(f"  → Ocean has negative elevation (e.g. -5000 m = 5000 m deep)")
    print(f"  → Land has positive elevation")
    ds.close()
else:
    print(f"No GEBCO NetCDF found in {GEBCO_PATH}")
    print("The model falls back to a synthetic flat grid — perfectly fine for testing.")
    print("")
    print("To get GEBCO data:")
    print("  1. Visit https://www.gebco.net/data_and_products/gridded_bathymetry_data/")
    print("  2. Register and download GEBCO_2024.nc (~2.7 GB)")
    print("  3. Place it in data/gebco_bathymetry/")
    print("Or use the downloader: python downloader.py --gebco  (manual steps only)")

### 1.3 Tidal Harmonic Constants — The Driving Force

Tides are reconstructed at the model's open boundaries from **harmonic constants** — amplitude $A_k$ and phase $\phi_k$ for each constituent — obtained from global tide models:

| Model | Source | Resolution | Format | Constituents | Status |
|-------|--------|------------|--------|-------------|--------|
| **GOT4.10c** | NASA GSFC | 0.5° | Per-constituent NetCDF (amplitude in cm, phase in °) | M2, S2, K1, O1, N2, K2, P1, Q1 | Recommended — no registration |
| **FES2014** | AVISO/LEGOS | 1/16° (~7 km) | Per-constituent NetCDF pairs (ocean + load) | M2, S2, K1, O1 + 30 more | Requires AVISO registration |
| **TPXO9** | Oregon State Univ. | 1/6° (~18 km) | Single multi-constituent NetCDF (real + imag parts) | M2, S2, K1, O1 + 11 more | Requires registration |

**GOT4.10c** is the default and recommended choice for this screening model:
- It's **free** — no registration required, direct download from NASA
- 0.5° resolution is appropriate for a 2 km screening model
- The `read_got_constituents()` function handles the cm→m and deg→rad conversions automatically
- File size is only ~44 MB (compressed tar)

FES2014 and TPXO9 provide higher resolution (useful for refined TELEMAC-2D runs) but require registration and are much larger (2–4 GB).

### 1.4 Land Mask Shapefiles — Where's the Coast?

To distinguish ocean from land, the model needs a **polygon of the coastline**. Two shapefiles are used:

| Shapefile | Source | Size | Purpose |
|-----------|--------|------|--------|
| **GADM Philippines** (gadm41_PHL_0.shp) | [gadm.org](https://gadm.org) | ~15 MB | Country boundary polygon — used to mask land cells |
| **OSM Philippines shoreline** (.shp.zip) | [Geofabrik](https://download.geofabrik.de/asia/philippines.html) | ~25 MB | Detailed coastline from OpenStreetMap |

The `build_land_mask()` function in `bathymetry.py` rasterises a shapefile onto the model grid using `rasterio.features.rasterize`:

```python
land_mask = build_land_mask(lon_1d, lat_1d, 'data/gadm41_PHL_shp/gadm41_PHL_0.shp')
# land_mask[j,i] == True  →  cell is land, excluded from computation
# land_mask[j,i] == False →  cell is ocean
```

If no shapefile is provided, the model falls back to a simple elevation-based mask: any cell where GEBCO elevation > 0 is treated as land. This works for open-ocean domains but misses inland water bodies and has jagged coastlines at coarse resolution.

### 1.5 The `downloader.py` Tool

The project ships with `downloader.py` in the repository root to automate data acquisition:

```bash
# Interactive mode — prompts for each dataset
python downloader.py

# Auto-download everything with direct URLs (OSM, GADM, GOT4.10c)
python downloader.py --all

# Download specific categories
python downloader.py --shoreline    # OSM + GADM shapefiles only
python downloader.py --tidal        # Show manual instructions for tidal data

# Custom output directory
python downloader.py --all --data-dir ./my_data
```

| Dataset | Auto-downloadable? | Notes |
|---------|-------------------|-------|
| GEBCO 2024 | ✗ (manual) | Requires licence acceptance — no direct URL |
| OSM Philippines shoreline | ✓ | Direct download from Geofabrik |
| GADM Philippines boundary | ✓ | Direct download from UC Davis |
| GOT4.10c harmonics | ✓ | Direct download from NASA GSFC |
| FES2014 harmonics | ✗ (manual) | Requires AVISO registration |
| TPXO9 harmonics | ✗ (manual) | Requires TPXO portal registration |

### 1.6 Expected `data/` Directory Structure

After running `downloader.py --all` and manually placing GEBCO, your `data/` directory should look like:

```
data/
├── gebco_bathymetry/
│   └── GEBCO_2024.nc                 # Manual download (~2.7 GB)
├── GOT4.10c/
│   ├── m2.nc                         # M2 amplitude + phase (cm, °)
│   ├── s2.nc
│   ├── k1.nc
│   ├── o1.nc
│   └── ...                           # N2, K2, P1, Q1 if available
├── gadm41_PHL_shp/
│   ├── gadm41_PHL_0.shp              # Country boundary
│   └── ...                           # .dbf, .shx, .prj companion files
├── philippines-latest-free.shp/
│   └── gis_osm_water_a_free_1.shp    # OSM coastline
├── got4.10c.tar.gz                   # Downloaded archive (~44 MB)
└── .gitkeep
```

The `config.yaml` connects these paths to the model:
```yaml
bathymetry:
  path: data/gebco_bathymetry/GEBCO_2024.nc
  land_shapefile: data/gadm41_PHL_shp/gadm41_PHL_0.shp

tidal_forcing:
  source: got
  path: data/GOT4.10c/grids_oceantide_netcdf/
  constituents: [M2, S2, K1, O1]
```

The model gracefully degrades if data is missing — synthetic flat bathymetry and uniform M2 forcing allow the solver to run and produce meaningful output for testing and development.

---
## 2. The Arakawa C-Grid

Our model uses a **staggered Arakawa C-grid**. Instead of storing all variables at cell centres, we store:

- **η** (free-surface elevation) at cell **centres**
- **u** (x-velocity) at **east/west faces** of each cell
- **v** (y-velocity) at **north/south faces** of each cell

This staggering avoids the checkerboard instability that plagues co-located grids and gives natural second-order accuracy for pressure-gradient and divergence calculations.

```
       v(j+1,i)          v(j+1,i+1)
          |                  |
    ------●------------------●------
          |                  |
   u(j,i) *   η(j,i)  u(j,i+1)  *  η(j,i+1)
          |                  |
    ------●------------------●------
          |                  |
       v(j,i)            v(j,i+1)

    ● = v-point (ny+1, nx)
    * = u-point (ny, nx+1)
    centre = η-point (ny, nx)
```

Let's create a simple uniform grid to see its attributes.

In [ ]:
# Create a small 10×8 grid at 5°N latitude, 2 km resolution
grid = StructuredGrid.from_uniform(
    nx=10, ny=8,
    dx=2000.0, dy=2000.0,  # 2 km spacing
    lat0=5.0               # Coriolis computed at this latitude
)

print(f"Grid shape (ny, nx): {grid.shape}")
print(f"dx = {grid.dx:.0f} m, dy = {grid.dy:.0f} m")
print(f"η shape:  {grid.ny} × {grid.nx}")
print(f"u  shape:  {grid.ny} × {grid.nx + 1}")
print(f"v  shape:  {grid.ny + 1} × {grid.nx}")
print(f"Coriolis f at 5°N: {grid.f[0,0]:.3e} rad/s")

### The `StructuredGrid` dataclass

All grid data lives in a single `StructuredGrid` dataclass. Key attributes:

| Attribute | Shape | Description |
|-----------|-------|-------------|
| `nx`, `ny` | scalar | Number of cells in x, y |
| `dx`, `dy` | scalar | Grid spacing [m] |
| `x`, `y` | (nx,), (ny,) | Cell-centre projected coordinates [m] |
| `lon`, `lat` | (ny, nx) | Cell-centre lat/lon in degrees |
| `h` | (ny, nx) | Bathymetric depth [m], positive down |
| `mask` | (ny, nx) | True = wet cell |
| `f` | (ny, nx) | Coriolis parameter at cell centres [rad/s] |
| `h_u` | (ny, nx+1) | Depth interpolated to u-points |
| `h_v` | (ny+1, nx) | Depth interpolated to v-points |
| `mask_u` | (ny, nx+1) | Wet mask at u-points |
| `mask_v` | (ny+1, nx) | Wet mask at v-points |
| `open_boundary` | (ny, nx) | True where η is prescribed |

Two factory methods create grids:
- `from_uniform()` — for idealised test cases
- `from_bathymetry()` — for real-world runs with GEBCO data

### Creating a grid from bathymetry

`from_bathymetry` takes 1D lon/lat arrays and a 2D depth field, plus an optional land mask. In production it works like this:

In [ ]:
# Simulate bathymetry: a 2° × 2° region at ~2 km resolution
nx, ny = 60, 60
lon_1d = np.linspace(120, 122, nx)
lat_1d = np.linspace(10, 12, ny)

# Create synthetic bathymetry with a deep channel
lon2d, lat2d = np.meshgrid(lon_1d, lat_1d)
depth = np.full((ny, nx), 100.0)  # flat 100 m base

# Carve a channel through the middle
mid_y = ny // 2
for j in range(ny):
    dist = abs(j - mid_y) * 2000.0  # approx metres from centreline
    if dist < 15000:
        depth[j, :] = 30.0 + dist * 0.005  # shallower near edges, deep in centre

# No land mask — all cells are wet
grid2 = StructuredGrid.from_bathymetry(
    lon_1d, lat_1d, depth,
    land_mask=None,
    min_depth=2.0
)

print(f"Grid: {grid2.nx}×{grid2.ny}, dx={grid2.dx:.0f} m, dy={grid2.dy:.0f} m")
print(f"Depth range: {grid2.h_min:.0f}–{grid2.h_max:.0f} m")
print(f"Wet cells: {grid2.mask.sum()} / {grid2.mask.size}")
print(f"Open boundary cells (perimeter): {grid2.open_boundary.sum()}")

### Land masking

In production, a **shapefile** (e.g. GADM Philippines boundary) is rasterised onto the grid. Cells that intersect the land polygon are marked as dry (`mask=False`). This is done in `bathymetry.build_land_mask()`, which uses `rasterio.features.rasterize`.

The open boundary is automatically set to **all wet cells on the domain perimeter** — these cells will have their surface elevation prescribed by tidal harmonics.

---
## 3. Bathymetry Loading & Regridding

The `bathymetry` module handles loading GEBCO NetCDF data and regridding to the model resolution.

### `load_gebco()`

Clips a large GEBCO NetCDF to a bounding box and returns 1D coordinate arrays + 2D elevation (positive up).

### `regrid_bathymetry()`

Coarsens GEBCO's 15-arcsecond (~450 m) resolution to the model's target resolution (typically 1–5 km) using bilinear interpolation via `scipy.interpolate.RegularGridInterpolator`.

In [ ]:
from model.bathymetry import regrid_bathymetry, elevation_to_depth

# Simulate a high-res input (like GEBCO at 15 arc-sec)
lon_high = np.linspace(120, 122, 400)
lat_high = np.linspace(10, 12, 400)
lonh, lath = np.meshgrid(lon_high, lat_high)
elev_high = -50.0 + 20.0 * np.sin(lonh * 10) * np.cos(lath * 10)  # fake GEBCO

# Regrid to ~2 km
lon_new, lat_new, elev_new = regrid_bathymetry(
    lon_high, lat_high, elev_high, resolution_km=2.0
)

print(f"High-res input: {len(lon_high)}×{len(lat_high)}")
print(f"Regridded to:    {len(lon_new)}×{len(lat_new)}")
print(f"Compression ratio: {(len(lon_high)*len(lat_high)) / (len(lon_new)*len(lat_new)):.0f}×")

# Convert elevation (positive up) to depth (positive down)
depth_new = elevation_to_depth(elev_new)
print(f"Depth range: {depth_new.min():.1f} – {depth_new.max():.1f} m")

---
## 4. Coriolis Parameter & Staggered-Grid Interpolation

### Coriolis

The Coriolis parameter $f = 2\Omega \sin(\phi)$ where $\Omega = 7.292 \times 10^{-5}$ rad/s and $\phi$ is latitude. It's strongest at the poles and zero at the equator. In the Philippines (4°–22°N), $f \approx 1–5 \times 10^{-5}$ rad/s — a moderate value.

In [ ]:
lats = np.array([0, 5, 10, 15, 20, 25, 90])
f_vals = coriolis(lats)
for lat, f in zip(lats, f_vals):
    print(f"  lat = {lat:3.0f}°  →  f = {f:.4e} rad/s")

### Staggered-grid interpolation

Because variables live at different locations, we need interpolation functions to move between them:

- **`interpolate_to_u(phi)`** → averages cell-centre φ to u-faces: `result[:, i] = (phi[:, i-1] + phi[:, i]) / 2`
- **`interpolate_to_v(phi)`** → averages cell-centre φ to v-faces: `result[j, :] = (phi[j-1, :] + phi[j, :]) / 2`
- **`v_at_u_pts(v)`** → averages surrounding v values at each u-point (4-point average)
- **`u_at_v_pts(u)`** → averages surrounding u values at each v-point (4-point average)
- **`velocity_at_centres(u, v)`** → averages u/v edge values to cell centres
- **`speed(u, v)`** → $|\mathbf{U}| = \sqrt{u_c^2 + v_c^2}$ at cell centres
- **`power_density(u, v)`** → $P = \frac{1}{2}\rho |\mathbf{U}|^3$ [W/m²]

In [ ]:
# Visualise the interpolation: create a test field and interpolate
ny, nx = 5, 6
h_test = np.arange(1, ny*nx+1).reshape(ny, nx).astype(float)
print("Cell-centre depth h (ny × nx):")
print(h_test)

h_u_test = interpolate_to_u(h_test)
print(f"\nAfter interpolate_to_u → shape {h_u_test.shape}:")
print(h_u_test)
print("(Notice: column i is average of h[:, i-1] and h[:, i])")

h_v_test = interpolate_to_v(h_test)
print(f"\nAfter interpolate_to_v → shape {h_v_test.shape}:")
print(h_v_test)
print("(Notice: row j is average of h[j-1, :] and h[j, :])")

---
## 5. Tidal Forcing

Tides drive the model by prescribing the free-surface elevation at **open boundary cells**. The model supports four sources:

| Source | Description | Resolution |
|--------|-------------|------------|
| `synthetic` | Uniform amplitude, single frequency (M2) | N/A |
| `got` | GOT4.10c — empirical, satellite altimetry | 0.5° |
| `fes2014` | FES2014 — finite-element, data-assimilated | 1/16° |
| `tpxo9` | TPXO9 — global inverse model | 1/6° |

### Tidal harmonics

Each **constituent** is defined by an amplitude $A_k$ and phase $\phi_k$ at each location. The free-surface elevation is reconstructed as:

$$\eta(t) = \sum_k A_k \cos(\omega_k t + \phi_k)$$

| Constituent | Period (hours) | $\omega$ (rad/s) | Origin |
|------------|---------|-------------------|--------|
| M2 | 12.42 | $1.405 \times 10^{-4}$ | Principal lunar semidiurnal |
| S2 | 12.00 | $1.454 \times 10^{-4}$ | Principal solar semidiurnal |
| K1 | 23.93 | $7.292 \times 10^{-5}$ | Lunisolar diurnal |
| O1 | 25.82 | $6.759 \times 10^{-5}$ | Principal lunar diurnal |

The four constituents above (M2, S2, K1, O1) capture ~80% of tidal variance globally.

In [ ]:
# Examine frequencies and periods of major constituents
for name in ["M2", "S2", "K1", "O1", "N2", "K2"]:
    omega = ASTRO_FREQUENCIES[name]
    period_hr = 2 * np.pi / omega / 3600.0
    period_day = period_hr / 24.0
    print(f"{name:4s}  ω = {omega:.6e} rad/s  |  T = {period_hr:6.2f} h  = {period_day:.4f} d")

### Synthetic tidal boundary

For quick testing, use `make_synthetic_tidal_boundary()` which applies uniform amplitude and zero phase to all boundary cells.

In [ ]:
# Create a synthetic boundary with M2 only, amplitude = 0.5 m
n_bnd = 20  # number of open boundary cells
synth_bnd = make_synthetic_tidal_boundary(
    n_boundary_cells=n_bnd,
    amplitude=0.5,
    constituents=["M2"]
)

print(f"Synthetic boundary: {len(synth_bnd.names)} constituent(s), {synth_bnd.n_boundary_cells} cells")
print(f"Amplitude shape: {synth_bnd.amp.shape}")
print(f"Phase shape:     {synth_bnd.phase.shape}")

# Evaluate at one time
t = 0.0
eta_t0 = synth_bnd.evaluate_at(t)
print(f"η at t=0: all = {eta_t0[0]:.3f} m (cos(0) = 1) ✓")

# Evaluate at quarter period (M2 period ≈ 44712 s)
T_m2 = 2 * np.pi / ASTRO_FREQUENCIES["M2"]
eta_quarter = synth_bnd.evaluate_at(T_m2 / 4)
print(f"η at t=T/4: all ≈ {eta_quarter[0]:.3f} m (cos(π/2) ≈ 0) ✓")

# Evaluate a full time series
t_series = np.linspace(0, 2 * T_m2, 200)
eta_series = synth_bnd.evaluate(t_series)  # shape (nt, n_bnd)
print(f"Time series shape: {eta_series.shape}")

### Reading real harmonic data

For production runs, the model reads harmonic constants from global datasets using:

- **`read_got_constituents()`** — GOT4.10c (per-constituent NetCDF, amplitude in cm, phase in degrees)
- **`read_fes_constituents()`** — FES2014 (ocean + load NetCDFs)
- **`read_tpxo_constituents()`** — TPXO9 (single NetCDF with real/imaginary parts)
- **`read_tidal_constituents()`** — dispatcher that chooses the right reader

All readers interpolate harmonic constants to the model's boundary cells using `scipy.interpolate.RegularGridInterpolator`.

### Spring-neap cycle from M2 + S2

When M2 and S2 are combined, they produce a **spring-neap cycle** (modulation every ~14.8 days). M2 alone gives constant-amplitude tides.

In [ ]:
# Compare M2-only vs M2+S2
bnd_m2 = make_synthetic_tidal_boundary(1, amplitude=0.5, constituents=["M2"])
bnd_m2s2 = make_synthetic_tidal_boundary(1, amplitude=0.5, constituents=["M2", "S2"])

t_days = np.linspace(0, 30, 5000)
t_sec = t_days * 86400.0

eta_m2 = bnd_m2.evaluate(t_sec)[:, 0]
eta_m2s2 = bnd_m2s2.evaluate(t_sec)[:, 0]

print("M2 only:       constant ±0.5 m envelope")
print(f"  max range = {eta_m2.max() - eta_m2.min():.2f} m")
print("M2+S2 spring-neap:")
print(f"  max range = {eta_m2s2.max() - eta_m2s2.min():.2f} m")
print("  Spring tides peak at ~1.0 m, neap tides ~0.0 m amplitude")
print("  This is why we run at least 15 days (one full spring-neap cycle)")

In [ ]:
# Quick plot of spring-neap modulation (matplotlib inline)
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(t_days, eta_m2, label="M2 only", linewidth=1)
ax1.set_ylabel("η [m]")
ax1.set_title("Tidal boundary elevation: M2 vs M2+S2")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(t_days, eta_m2s2, label="M2 + S2", linewidth=1, color="C1")
ax2.set_xlabel("Time [days]")
ax2.set_ylabel("η [m]")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. The Shallow-Water Solver

`ShallowWaterSolver` implements a **forward-backward time-stepping scheme** on the Arakawa C-grid. Let's examine each piece.

### Governing equations

The depth-averaged shallow-water equations (in Cartesian coordinates for the local grid):

**Momentum (x):**
$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} = -g\frac{\partial \eta}{\partial x} + f v - \frac{C_d}{H} u\sqrt{u^2 + v^2} + A_h \nabla^2 u$$

**Momentum (y):**
$$\frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} = -g\frac{\partial \eta}{\partial y} - f u - \frac{C_d}{H} v\sqrt{u^2 + v^2} + A_h \nabla^2 v$$

**Continuity:**
$$\frac{\partial \eta}{\partial t} + \frac{\partial}{\partial x}(Hu) + \frac{\partial}{\partial y}(Hv) = 0$$

where $H = h + \eta$ is total water depth, $C_d$ is bottom drag, $A_h$ is horizontal eddy viscosity.

### Forward-backward time-stepping

The solver advances one time step $\Delta t$ as:

1. **Evaluate tidal BC** — set η at open boundary cells
2. **u-momentum** — compute all tendencies, apply semi-implicit friction
3. **v-momentum** — compute all tendencies, apply semi-implicit friction
4. **Continuity** — update η from flux divergence (at interior cells only)

This is a **forward-backward** scheme (also called a **leapfrog**-like scheme): momentum is advanced from $t$ to $t+\Delta t$ using η at $t$, then continuity uses the **new** velocities to step η to $t+\Delta t$. This is second-order accurate in time for the wave propagation terms.

### Semi-implicit bottom friction

Friction is treated semi-implicitly to avoid instability in shallow water:

$$u^{n+1} = \frac{u^n + \Delta t \cdot (\text{other terms})}{1 + \Delta t \cdot C_d \cdot |\mathbf{U}| / H}$$

This ensures friction can only reduce velocity, never reverse it.

In [ ]:
# --- Build a complete mini-model: 1D tidal channel ---
L = 50000.0    # channel length [m]
H = 30.0       # constant depth [m]
nx = 40
ny = 3
dx = L / nx
dy = dx

grid_ch = StructuredGrid.from_uniform(nx=nx, ny=ny, dx=dx, dy=dy, lat0=0.0)
grid_ch.h[:, :] = H
grid_ch.h_u[:] = H
grid_ch.h_v[:] = H
grid_ch.mask[:] = True
grid_ch.mask_u[:] = True
grid_ch.mask_v[:] = True
grid_ch.open_boundary[:, 0] = True   # left open
grid_ch.open_boundary[:, -1] = True  # right open
grid_ch.f[:] = 0.0
grid_ch.f_u[:] = 0.0
grid_ch.f_v[:] = 0.0

print(f"Channel: {nx}×{ny}, dx={dx:.0f}m, dy={dy:.0f}m")
print(f"Length = {L/1000:.1f} km, Depth = {H:.0f} m")
print(f"Open boundaries at x=0 and x={L/1000:.1f} km")

In [ ]:
# Create solver and set M2 tidal forcing
solver_ch = ShallowWaterSolver(
    grid_ch,
    cd=0.0025,      # bottom drag
    ah=0.0,          # no horizontal mixing
    advection=False   # linear only
)

amp = 0.5
omega = ASTRO_FREQUENCIES["M2"]

def eta_bc(t):
    """Prescribe η = amp * cos(ωt) at all open boundary cells."""
    bc = np.zeros_like(grid_ch.eta)
    bc[grid_ch.open_boundary] = amp * np.cos(omega * t)
    return bc

solver_ch.set_open_boundary_eta(eta_bc)

# Run for 3 M2 periods
T_m2 = 2 * np.pi / omega
dt_ch = 3.0  # small time step for accuracy
duration_ch = 3 * T_m2

print(f"Running for {duration_ch/3600:.1f} hours ({duration_ch/T_m2:.1f} M2 periods)...")
print(f"dt = {dt_ch:.1f} s, {int(duration_ch/dt_ch)} steps")

# Collect snapshots
snapshots_ch = []
def snap_cb(solv, step):
    if step % 50 == 0:
        snapshots_ch.append({
            "t": solv.time,
            "eta": solv.eta.copy(),
            "u": solv.u.copy(),
            "v": solv.v.copy(),
        })
    return None

solver_ch.run(dt=dt_ch, duration=duration_ch, callback=snap_cb, progress_interval=duration_ch)
print(f"Collected {len(snapshots_ch)} snapshots")

In [ ]:
# Plot the wave propagation along the channel
j_mid = ny // 2

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Space-time η
eta_xt = np.array([s["eta"][j_mid, :] for s in snapshots_ch])
t_arr = np.array([s["t"] for s in snapshots_ch]) / 3600
x_arr = np.arange(nx) * dx / 1000

im = axes[0].pcolormesh(x_arr, t_arr, eta_xt, shading='auto', cmap='RdBu_r')
axes[0].set_xlabel("x [km]")
axes[0].set_ylabel("Time [hours]")
axes[0].set_title("η(x,t) — free surface")
plt.colorbar(im, ax=axes[0], label="η [m]")

# η at channel midpoint vs time
xc_mid = nx // 2
axes[1].plot(t_arr, [s["eta"][j_mid, xc_mid] for s in snapshots_ch], 'b-', linewidth=1)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel("Time [hours]")
axes[1].set_ylabel("η [m]")
axes[1].set_title("η at channel midpoint")
axes[1].grid(True, alpha=0.3)

# u at channel midpoint vs time
axes[2].plot(t_arr, [s["u"][j_mid, xc_mid] for s in snapshots_ch], 'r-', linewidth=1)
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel("Time [hours]")
axes[2].set_ylabel("u [m/s]")
axes[2].set_title("u at channel midpoint")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Examining each term in `step()`

Let's isolate what each physics term contributes to the momentum equations. The `step()` method computes:

```python
du_dt = -G * dpdx           # pressure gradient
      + f_u * v_interp       # Coriolis
      - C_d * u * |U| / H    # bottom friction
      + ah * ∇²u             # horizontal mixing (optional)
      + advection_u          # nonlinear advection (optional)
```

Then applies semi-implicit friction: `u_new = (u + dt*du_dt) / (1 + dt*C_d*|U|/H)`

In [ ]:
# Compare: with vs without bottom friction
print("\n=== Effect of bottom drag C_d ===")

def run_channel_with_cd(cd, label, duration_days=0.2):
    """Run a short channel simulation and return peak velocity."""
    solver = ShallowWaterSolver(grid_ch, cd=cd, ah=0.0, advection=False)
    solver.set_open_boundary_eta(eta_bc)
    solver.run(dt=5.0, duration=duration_days*86400, progress_interval=999999)
    spd = speed(solver.u, solver.v)
    return float(np.max(spd))

for cd in [0.0, 0.001, 0.0025, 0.005, 0.01]:
    u_max = run_channel_with_cd(cd, f"C_d={cd}")
    print(f"  C_d = {cd:.4f}  →  |U|_max = {u_max:.3f} m/s")

In [ ]:
# Compare: with vs without Coriolis
print("\n=== Effect of Coriolis ===")

# Rebuild grid with Coriolis at 12°N
grid_f = StructuredGrid.from_uniform(nx=nx, ny=ny, dx=dx, dy=dy, lat0=12.0)
grid_f.h[:, :] = H
grid_f.h_u[:] = H
grid_f.h_v[:] = H
grid_f.mask[:] = True
grid_f.mask_u[:] = True
grid_f.mask_v[:] = True
grid_f.open_boundary[:, 0] = True
grid_f.open_boundary[:, -1] = True
# Don't zero out f — keep Coriolis

solver_f = ShallowWaterSolver(grid_f, cd=0.0025, ah=0.0, advection=False)

def eta_bc_f(t):
    bc = np.zeros_like(grid_f.eta)
    bc[grid_f.open_boundary] = 0.5 * np.cos(omega * t)
    return bc

solver_f.set_open_boundary_eta(eta_bc_f)
solver_f.run(dt=5.0, duration=0.3*86400, progress_interval=999999)

# Check v-velocity (should develop due to Coriolis)
v_max = float(np.max(np.abs(solver_f.v)))
print(f"  Coriolis at 12°N: f = {grid_f.f[0,0]:.3e} rad/s")
print(f"  max |v| = {v_max:.3f} m/s  (non-zero due to Coriolis turning)")
print(f"  Without Coriolis, v would be ~0 in a purely x-forced channel")

In [ ]:
# Compare: linear vs nonlinear (advection enabled)
print("\n=== Effect of nonlinear advection ===")

solver_lin = ShallowWaterSolver(grid_ch, cd=0.0025, ah=0.0, advection=False)
solver_lin.set_open_boundary_eta(eta_bc)
solver_lin.run(dt=5.0, duration=0.3*86400, progress_interval=999999)
u_max_lin = float(np.max(np.abs(solver_lin.u)))

solver_nl = ShallowWaterSolver(grid_ch, cd=0.0025, ah=0.0, advection=True)
solver_nl.set_open_boundary_eta(eta_bc)
solver_nl.run(dt=5.0, duration=0.3*86400, progress_interval=999999)
u_max_nl = float(np.max(np.abs(solver_nl.u)))

print(f"  Linear:     max |u| = {u_max_lin:.4f} m/s")
print(f"  Nonlinear:  max |u| = {u_max_nl:.4f} m/s")
print(f"  Difference = {abs(u_max_nl - u_max_lin):.4f} m/s  ({100*abs(u_max_nl-u_max_lin)/max(u_max_lin,0.001):.2f}%)")
print("  Advection matters more in narrow straits with strong currents")

In [ ]:
# Compare: with vs without horizontal mixing (eddy viscosity)
print("\n=== Effect of horizontal eddy viscosity A_h ===")

for ah in [0.0, 1.0, 10.0, 100.0]:
    s = ShallowWaterSolver(grid_ch, cd=0.0025, ah=ah, advection=False)
    s.set_open_boundary_eta(eta_bc)
    s.run(dt=5.0, duration=0.3*86400, progress_interval=999999)
    u_max = float(np.max(np.abs(s.u)))
    print(f"  A_h = {ah:6.1f} m²/s  →  max |u| = {u_max:.4f} m/s")

---
## 7. CFL Stability Condition

The model is **explicit**, which means the time step must satisfy the Courant-Friedrichs-Lewy condition:

$$\Delta t \leq \frac{\Delta x_{\text{eff}}}{\sqrt{g \, h_{\text{max}}}}$$

where $\Delta x_{\text{eff}} = 1 / \sqrt{1/\Delta x^2 + 1/\Delta y^2}$ is the effective 2D grid spacing, $h_{\text{max}}$ is the maximum depth, and a safety factor (default 0.5) is applied.

If $\Delta t$ is too large, the model will produce NaN or blow up.

In [ ]:
# Demonstrate CFL computation for different resolutions and depths
resolutions_km = [0.5, 1.0, 2.0, 5.0, 10.0]
depths_m = [10, 50, 200, 1000, 5000]

print("CFL time step [s] for different resolutions and depths (safety=0.5):\n")
print(f"{'':>12s}", end="")
for h in depths_m:
    print(f"{'h=' + str(h) + 'm':>12s}", end="")
print()

for res in resolutions_km:
    dx = res * 1000.0
    print(f"{'Δx=' + str(res) + 'km':>12s}", end="")
    for h in depths_m:
        dt = cfl_timestep(dx, dx, h, safety=0.5)
        print(f"{dt:12.1f}", end="")
    print()

In [ ]:
# What happens if we violate CFL? (Deliberately run with dt too large)
print("Testing CFL violation on a small basin...")

grid_test = StructuredGrid.from_uniform(nx=20, ny=15, dx=1000.0, dy=1000.0, lat0=0.0)
grid_test.h[:, :] = 50.0
grid_test.h_u[:] = 50.0
grid_test.h_v[:] = 50.0
grid_test.mask[:] = True
grid_test.mask_u[:] = True
grid_test.mask_v[:] = True
grid_test.f[:] = 0.0
grid_test.f_u[:] = 0.0
grid_test.f_v[:] = 0.0

x_c = np.arange(20) * 1000 + 500
eta0 = 0.1 * np.sin(2 * np.pi * x_c / 20000.0)
eta0_2d = np.tile(eta0, (15, 1))

# Safe run
dt_safe = cfl_timestep(1000.0, 1000.0, 50.0, safety=0.5)
print(f"  Safe dt (CFL=0.5): {dt_safe:.1f} s")

s_safe = ShallowWaterSolver(grid_test, cd=0.0, ah=0.0, advection=False)
s_safe.set_initial_conditions(eta0=eta0_2d)
s_safe.run(dt=dt_safe, duration=3600, progress_interval=999999)
eta_max_safe = float(np.max(np.abs(s_safe.eta)))
print(f"  max |η| after 1h with safe dt: {eta_max_safe:.4f} m (stable)")

# Unsafe run
dt_unsafe = dt_safe * 10
print(f"  Unsafe dt (10×): {dt_unsafe:.1f} s")

s_unsafe = ShallowWaterSolver(grid_test, cd=0.0, ah=0.0, advection=False)
s_unsafe.set_initial_conditions(eta0=eta0_2d)
has_nan = False
try:
    s_unsafe.run(dt=dt_unsafe, duration=3600, progress_interval=999999)
    if np.any(np.isnan(s_unsafe.eta)):
        has_nan = True
except Exception as e:
    print(f"  Exception: {e}")

if has_nan or np.any(np.isnan(s_unsafe.eta)):
    print("  → NaN detected! Model blew up.")
else:
    eta_max_unsafe = float(np.max(np.abs(s_unsafe.eta)))
    print(f"  max |η| after 1h with unsafe dt: {eta_max_unsafe:.4f} m")

---
## 8. Running the Model

The `run()` method on `ShallowWaterSolver` orchestrates the simulation:

```python
solver.run(dt=10.0, duration=15*86400, callback=my_callback, progress_interval=3600)
```

- `dt` — time step [s] (auto-computed via CFL in production)
- `duration` — total simulation time [s]
- `callback` — function `f(solver, step_n) -> None | dict` called each step
- `progress_interval` — how often to log progress (default: every 3600 s)

### Production run flow

The full `run.py` pipeline:

1. Load GEBCO bathymetry → clip to domain → regrid → depth conversion → land mask
2. Build grid via `StructuredGrid.from_bathymetry()`
3. Read tidal harmonics (or use synthetic) → `TidalBoundary`
4. Initialise `ShallowWaterSolver` with config parameters
5. Compute CFL-limited time step
6. Run with a snapshot callback that saves every N hours
7. Mass conservation check (drift should be < 0.01%)
8. Write outputs: NetCDF, GeoTIFF, GeoJSON

In [ ]:
# Demonstrate mass conservation check
grid_cons = StructuredGrid.from_uniform(nx=20, ny=20, dx=1000.0, dy=1000.0, lat0=0.0)
grid_cons.h[:, :] = 40.0
grid_cons.h_u[:] = 40.0
grid_cons.h_v[:] = 40.0
grid_cons.mask[:] = True
grid_cons.mask_u[:] = True
grid_cons.mask_v[:] = True
grid_cons.open_boundary[:] = False  # closed basin
grid_cons.f[:] = 0.0
grid_cons.f_u[:] = 0.0
grid_cons.f_v[:] = 0.0

# Initial Gaussian bump
x_c = np.arange(20) * 1000 + 500
y_c = np.arange(20) * 1000 + 500
yy, xx = np.meshgrid(y_c, x_c, indexing="ij")
eta0 = 0.5 * np.exp(-((xx-10000)**2 + (yy-10000)**2) / (2*3000**2))

solver_cons = ShallowWaterSolver(grid_cons, cd=0.0, ah=0.0, advection=False)
solver_cons.set_initial_conditions(eta0=eta0)

vol0 = solver_cons.total_volume()
print(f"Initial volume: {vol0:.3e} m³")

solver_cons.run(dt=1.0, duration=7200, progress_interval=999999)

vol1 = solver_cons.total_volume()
drift_pct = 100 * abs(vol1 - vol0) / vol0
print(f"Final volume:   {vol1:.3e} m³")
print(f"Drift:          {drift_pct:.6f}%  (should be << 0.01%)")

---
## 9. Output: NetCDF, GeoTIFF, Hotspot GeoJSON

The `output` module generates three types of results:

### NetCDF (`results.nc`)
Contains full time series: η(t,x,y), u(t,x,y), v(t,x,y), power_density(t,x,y). This is the primary scientific output for post-processing.

### GeoTIFF (`tidal_power_density.tif`)
Cloud-Optimised GeoTIFF of **time-mean** power density $\bar{P}(x,y) = \frac{1}{T}\int_0^T \frac{1}{2}\rho |\mathbf{U}|^3 dt$. This is what the web map displays as an overlay.

### GeoJSON (`hotspots.geojson`)
Point features for every cell where $\bar{P} \geq$ hotspot threshold (default 200 W/m²). Used to highlight potential tidal-energy sites.

In [ ]:
# Demonstrate output creation with synthetic data
import tempfile

grid_out = StructuredGrid.from_uniform(nx=10, ny=8, dx=2000.0, dy=2000.0, lat0=12.0)
grid_out.h[:, :] = 50.0
grid_out.mask[:, :] = True

# Fake time series: 24 hours of data
nt = 24
times = np.arange(nt) * 3600.0
eta_hist = 0.5 * np.sin(2*np.pi*times[:,None,None]/44712.0) * np.ones((nt,8,10))
u_hist = 0.3 * np.cos(2*np.pi*times[:,None,None]/44712.0) * np.ones((nt,8,11))
v_hist = 0.1 * np.sin(2*np.pi*times[:,None,None]/44712.0) * np.ones((nt,9,10))
power_hist = 0.5 * 1025.0 * np.sqrt(u_hist[:,:,1:]**2 + v_hist[:,1:,:]**2) ** 3

# Create xarray Dataset
ds = create_results_dataset(grid_out, times, eta_hist, u_hist, v_hist, power_hist)
print("Dataset:")
print(ds)

# Write NetCDF
tmpdir = tempfile.mkdtemp()
nc_path = os.path.join(tmpdir, "results.nc")
write_netcdf(ds, nc_path)
print(f"\nWrote NetCDF: {nc_path}")

# Write GeoTIFF (mean power density)
power_mean = np.mean(power_hist, axis=0)
tif_path = os.path.join(tmpdir, "power.tif")
try:
    write_mean_power_geotiff(grid_out, power_mean, tif_path)
    print(f"Wrote GeoTIFF: {tif_path}")
except ImportError:
    print("Skipped GeoTIFF: rasterio not available")

# Write hotspots GeoJSON
geojson_path = os.path.join(tmpdir, "hotspots.geojson")
write_hotspots_geojson(grid_out, power_mean, threshold=5.0, path=geojson_path)
print(f"Wrote GeoJSON: {geojson_path}")

In [ ]:
# Inspect the GeoJSON
import json
with open(geojson_path) as f:
    geojson = json.load(f)
print(f"Hotspots: {len(geojson['features'])} features")
for feat in geojson['features'][:3]:
    print(f"  lon={feat['geometry']['coordinates'][0]:.3f}, "
          f"lat={feat['geometry']['coordinates'][1]:.3f}, "
          f"P={feat['properties']['power_density_Wm2']:.1f} W/m², "
          f"h={feat['properties']['depth_m']:.0f} m")

---
## 10. Parameter Sensitivity

Let's systematically explore how each parameter affects the model output:

In [ ]:
# ---- Setup a reusable test channel ----
def build_channel_grid(L=50000.0, H=30.0, nx=40, ny=3, include_coriolis=False, lat0=0.0):
    dx = L / nx
    dy = dx
    grid = StructuredGrid.from_uniform(nx=nx, ny=ny, dx=dx, dy=dy, lat0=lat0)
    grid.h[:, :] = H
    grid.h_u[:] = H
    grid.h_v[:] = H
    grid.mask[:] = True
    grid.mask_u[:] = True
    grid.mask_v[:] = True
    grid.open_boundary[:, 0] = True
    grid.open_boundary[:, -1] = True
    if not include_coriolis:
        grid.f[:] = 0.0
        grid.f_u[:] = 0.0
        grid.f_v[:] = 0.0
    return grid

def run_and_get_power(grid, cd=0.0025, ah=0.0, advection=False, duration_days=0.5):
    """Run a channel simulation and return mean power density."""
    amp = 0.5
    omega = ASTRO_FREQUENCIES["M2"]
    
    def eta_bc(t):
        bc = np.zeros_like(grid.eta)
        bc[grid.open_boundary] = amp * np.cos(omega * t)
        return bc
    
    solver = ShallowWaterSolver(grid, cd=cd, ah=ah, advection=advection)
    solver.set_open_boundary_eta(eta_bc)
    
    # Collect power snapshots
    powers = []
    def cb(s, step):
        if step % 10 == 0:
            powers.append(s.compute_power_density())
        return None
    
    solver.run(dt=5.0, duration=duration_days*86400, callback=cb, progress_interval=999999)
    
    if powers:
        mean_power = np.mean(powers, axis=0)
        return float(np.max(mean_power))
    else:
        return 0.0

In [ ]:
# --- Sensitivity 1: Bottom drag C_d ---
print("=== Sensitivity: Bottom drag coefficient C_d ===\n")
grid_s1 = build_channel_grid()
cd_vals = [0.0, 0.0005, 0.001, 0.0025, 0.005, 0.01, 0.02]
p_cd = []
for cd in cd_vals:
    p = run_and_get_power(grid_s1, cd=cd)
    p_cd.append(p)
    print(f"  C_d = {cd:.4f}  →  max P = {p:.2f} W/m²")

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(cd_vals, p_cd, 'o-', markersize=6)
ax.set_xlabel("Bottom drag coefficient C_d")
ax.set_ylabel("Max mean power density [W/m²]")
ax.set_title("Sensitivity: higher drag → lower velocities → less power")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Sensitivity 2: Channel depth ---
print("\n=== Sensitivity: Channel depth H ===\n")
depth_vals = [5, 10, 20, 30, 50, 100, 200]
p_depth = []
for H in depth_vals:
    grid_d = build_channel_grid(H=H)
    p = run_and_get_power(grid_d, cd=0.0025)
    p_depth.append(p)
    print(f"  H = {H:4.0f} m  →  max P = {p:.2f} W/m²")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depth_vals, p_depth, 'o-', markersize=6)
ax.set_xlabel("Depth H [m]")
ax.set_ylabel("Max mean power density [W/m²]")
ax.set_title("Sensitivity: shallower water → larger η/u ratio → more complex response")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Sensitivity 3: Grid resolution ---
print("\n=== Sensitivity: Grid resolution ===\n")
from model.utils import cfl_timestep

resolutions_m = [500, 1000, 2000, 5000, 10000]
p_res = []
for dx in resolutions_m:
    nx = max(5, int(50000 / dx))
    grid_r = build_channel_grid(L=50000, H=30, nx=nx, ny=3)
    p = run_and_get_power(grid_r, cd=0.0025)
    p_res.append(p)
    dt_cfl = cfl_timestep(dx, dx, 30, safety=0.5)
    print(f"  Δx = {dx:5.0f} m ({dx/1000:.1f} km), nx={nx:3d}, dt_CFL={dt_cfl:.1f}s  →  max P = {p:.2f} W/m²")

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(resolutions_m, p_res, 'o-', markersize=6)
ax.set_xlabel("Grid spacing Δx [m]")
ax.set_ylabel("Max mean power density [W/m²]")
ax.set_title("Sensitivity: coarser grids may miss narrow-channel accelerations")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Sensitivity 4: Tidal amplitude ---
print("\n=== Sensitivity: Tidal amplitude ===\n")

def run_with_amp(grid, amp, cd=0.0025):
    omega = ASTRO_FREQUENCIES["M2"]
    def eta_bc(t):
        bc = np.zeros_like(grid.eta)
        bc[grid.open_boundary] = amp * np.cos(omega * t)
        return bc
    solver = ShallowWaterSolver(grid, cd=cd, ah=0.0, advection=False)
    solver.set_open_boundary_eta(eta_bc)
    powers = []
    def cb(s, step):
        if step % 10 == 0:
            powers.append(s.compute_power_density())
        return None
    solver.run(dt=5.0, duration=0.5*86400, callback=cb, progress_interval=999999)
    return float(np.max(np.mean(powers, axis=0))) if powers else 0.0

grid_amp = build_channel_grid()
amp_vals = [0.1, 0.2, 0.5, 1.0, 1.5, 2.0]
p_amp = []
for a in amp_vals:
    p = run_with_amp(grid_amp, a)
    p_amp.append(p)
    print(f"  Amplitude = {a:.1f} m  →  max P = {p:.2f} W/m²")

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(amp_vals, p_amp, 'o-', markersize=6)
ax.set_xlabel("Tidal amplitude [m]")
ax.set_ylabel("Max mean power density [W/m²]")
ax.set_title("Sensitivity: P ∝ U³, with a roughly cubic relationship to amplitude")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Sensitivity 5: Simulation duration (spin-up) ---
print("\n=== Sensitivity: Simulation duration (spin-up effect) ===\n")
grid_spin = build_channel_grid()

def run_duration(grid, days):
    omega = ASTRO_FREQUENCIES["M2"]
    def eta_bc(t):
        bc = np.zeros_like(grid.eta)
        bc[grid.open_boundary] = 0.5 * np.cos(omega * t)
        return bc
    solver = ShallowWaterSolver(grid, cd=0.0025, ah=0.0, advection=False)
    solver.set_open_boundary_eta(eta_bc)
    powers = []
    def cb(s, step):
        if step % 10 == 0:
            powers.append(s.compute_power_density())
        return None
    solver.run(dt=5.0, duration=days*86400, callback=cb, progress_interval=999999)
    return np.array(powers)

# Run a longer simulation and track convergence
powers_spin = run_duration(grid_spin, 3.0)  # 3 days
max_per_snap = np.max(powers_spin, axis=(1,2))
t_snap = np.arange(len(max_per_snap)) * 5.0 * 10 / 3600  # hours (every 10 steps of dt=5s)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_snap, max_per_snap, linewidth=1)
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Max power density [W/m²]")
ax.set_title("Power density convergence: model reaches quasi-steady state after ~1 M2 period (12.4 h)")
ax.axvline(12.42, color='red', linestyle='--', alpha=0.5, label='1 M2 period')
ax.axvline(24.84, color='orange', linestyle='--', alpha=0.5, label='2 M2 periods')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("The model reaches a quasi-steady oscillatory state after ~1-2 tidal periods.")
print("For production runs, the first few periods can be discarded as spin-up.")

---
## 11. Validation Tests

The test suite validates the solver against analytical solutions for three canonical cases:

### 11a. Seiche (Standing Wave) — Merian's Formula

In a closed rectangular basin of length $L$ and depth $h$, the fundamental seiche period is:

$$T = \frac{2L}{\sqrt{g h}}$$

An initial free-surface tilt should oscillate at exactly this period.

In [ ]:
# Validate seiche period against Merian's formula
L_basin = 10000.0
H_basin = 10.0
g = 9.81

T_merian_expected = 2 * L_basin / np.sqrt(g * H_basin)
print(f"Merian period: T = {T_merian_expected:.1f} s  ({T_merian_expected/60:.1f} min)")

# Build closed basin
nx = 50
ny = 3
dx = L_basin / nx
dy = dx

grid_seiche = StructuredGrid.from_uniform(nx=nx, ny=ny, dx=dx, dy=dy, lat0=0.0)
grid_seiche.h[:, :] = H_basin
grid_seiche.h_u[:] = H_basin
grid_seiche.h_v[:] = H_basin
grid_seiche.mask[:] = True
grid_seiche.mask_u[:] = True
grid_seiche.mask_v[:] = True
grid_seiche.open_boundary[:] = False
grid_seiche.f[:] = 0.0
grid_seiche.f_u[:] = 0.0
grid_seiche.f_v[:] = 0.0

# Initial condition: cos(πx/L) — first-mode seiche
x_c = np.arange(nx) * dx + dx/2
eta0 = 0.1 * np.cos(np.pi * x_c / L_basin)
eta0_2d = np.tile(eta0, (ny, 1))

solver_seiche = ShallowWaterSolver(grid_seiche, cd=0.0, ah=0.0, advection=False)
solver_seiche.set_initial_conditions(eta0=eta0_2d)

# Track zero crossings to measure period
dt = 0.1
eta_history_seiche = []
t_history_seiche = []

duration = 3 * T_merian_expected
n_steps = int(duration / dt)

for i in range(n_steps):
    solver_seiche.step(dt)
    if i % 20 == 0:
        eta_history_seiche.append(solver_seiche.eta[1, nx//2])
        t_history_seiche.append(solver_seiche.time)

# Find zero crossings
eta_arr = np.array(eta_history_seiche)
t_arr = np.array(t_history_seiche)
crossings = []
for i in range(1, len(eta_arr)):
    if eta_arr[i-1] * eta_arr[i] <= 0:
        crossings.append(t_arr[i])

if len(crossings) >= 3:
    measured_T = crossings[2] - crossings[0]
    error_pct = 100 * abs(measured_T - T_merian_expected) / T_merian_expected
    print(f"Measured period: {measured_T:.1f} s")
    print(f"Analytical:      {T_merian_expected:.1f} s")
    print(f"Error:           {error_pct:.2f}%  ← should be < 10%")
else:
    print(f"Not enough zero crossings ({len(crossings)}) — need a longer run")

In [ ]:
# Plot the seiche oscillation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Time series at midpoint
ax1.plot(np.array(t_history_seiche)/60, eta_arr, 'b-', linewidth=1)
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel("Time [minutes]")
ax1.set_ylabel("η at midpoint [m]")
ax1.set_title("Seiche oscillation at basin centre")
ax1.grid(True, alpha=0.3)

# Snapshots along x at different times
from matplotlib.cm import viridis
n_snap = 6
step_snap = len(eta_history_seiche) // n_snap
for i in range(n_snap):
    idx = i * step_snap
    ax2.plot(x_c/1000, eta_history_seiche[idx][1,:], 
             label=f"t={t_history_seiche[idx]/60:.0f} min", linewidth=1)
ax2.set_xlabel("x [km]")
ax2.set_ylabel("η [m]")
ax2.set_title("η(x) snapshots — standing wave pattern")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 11b. Tidal Channel — Phase Relationship

In a frictionless channel, velocity should lead elevation by approximately 90° (velocity is in phase with the pressure gradient, which is the gradient of η). With friction, the phase lead decreases.

In [ ]:
# Phase relationship in the tidal channel simulation from section 6
eta_mid = np.array([s["eta"][j_mid, xc_mid] for s in snapshots_ch])
u_mid = np.array([s["u"][j_mid, xc_mid] for s in snapshots_ch])

# Cross-correlation to find phase shift
from scipy import signal
corr = signal.correlate(u_mid - np.mean(u_mid), eta_mid - np.mean(eta_mid), mode='same')
lags = signal.correlation_lags(len(u_mid), len(eta_mid), mode='same')
peak_lag = lags[np.argmax(corr)]
dt_snap = np.mean(np.diff(t_arr * 3600))
phase_shift_rad = peak_lag * ASTRO_FREQUENCIES["M2"] * dt_snap
phase_shift_deg = np.rad2deg(phase_shift_rad)

print(f"Phase shift: {phase_shift_rad:.2f} rad = {phase_shift_deg:.1f}°")
print(f"(u leads η by {abs(phase_shift_rad):.2f} rad)")
print(f"Expected: ~π/2 (90°) for frictionless, less for frictional")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_arr, eta_mid/np.max(np.abs(eta_mid)), 'b-', label='η (normalised)', linewidth=1.5)
ax.plot(t_arr, u_mid/np.max(np.abs(u_mid)), 'r-', label='u (normalised)', linewidth=1.5)
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Normalised")
ax.set_title(f"Velocity leads elevation by {phase_shift_deg:.0f}° (C_d=0.0025)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### 11c. Mass Conservation

In a closed basin with no open boundaries, total volume must remain constant (to within numerical truncation). The solver's `total_volume()` method computes $\sum (h + \eta) \cdot \Delta x \Delta y$ over all wet cells. Production runs show drift < 0.01%.

---
## Summary: The Complete Model Pipeline

```
        ┌─────────────────────────────────────────────────┐
        │                  run.py                          │
        │  ┌──────────────────────────────────────────┐   │
Data ──┼──┤ 1. Load GEBCO → clip → regrid → depth    │   │
        │  │    → land mask → StructuredGrid          │   │
        │  ├──────────────────────────────────────────┤   │
Data ──┼──┤ 2. Read tidal harmonics → TidalBoundary   │   │
        │  ├──────────────────────────────────────────┤   │
        │  │ 3. ShallowWaterSolver(grid, cd, ah, ...) │   │
        │  │    ├─ set_initial_conditions()           │   │
        │  │    ├─ set_open_boundary_eta()            │   │
        │  │    └─ run(dt, duration, callback)        │   │
        │  │         ├─ step() × N_steps              │   │
        │  │         │    ├─ apply_eta_bc()           │   │
        │  │         │    ├─ u-momentum               │   │
        │  │         │    ├─ v-momentum               │   │
        │  │         │    └─ continuity (η update)    │   │
        │  │         └─ callback → snapshots          │   │
        │  ├──────────────────────────────────────────┤   │
        │  │ 4. Mass conservation check               │   │
        │  ├──────────────────────────────────────────┤   │
Output──┼──┤ 5. Write NetCDF, GeoTIFF, GeoJSON        │   │
        │  └──────────────────────────────────────────┘   │
        └─────────────────────────────────────────────────┘
```

### Key takeaways

| Parameter | Effect |
|-----------|--------|
| **C_d** (bottom drag) | Higher drag reduces velocities → lower power (P ∝ U³) |
| **Grid resolution** | Finer grids capture narrow-channel accelerations, but increase computation cost |
| **Tidal amplitude** | Larger tides → stronger currents → exponentially more power (cubic) |
| **Water depth** | Shallower water amplifies η response; deeper water transmits energy efficiently |
| **CFL safety** | Lower safety factor (0.25 vs 0.5) avoids NaN in complex bathymetry |
| **Coriolis** | Turns flow to the right (NH); important for wide domains (Philippines ~5°–20°N) |
| **Advection** | Matters in constrictions where velocity gradients are large |
| **A_h** (viscosity) | Smooths grid-scale noise; 0 is fine for well-resolved flows |

For the Philippine archipelago, typical hotspots (San Bernardino Strait, Surigao Strait) have power densities exceeding 200–500 W/m², driven by strong tidal currents (2–4 m/s) through narrow passages.